In [ ]:
import torch
from tqdm import tqdm
from datasets import load_dataset, Dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
MODELNAME = "roberta/checkpoint-750"
# MODELNAME = "roberta-augmented/checkpoint-1215"
# MODELNAME = "bert/checkpoint-750"
# MODELNAME = "bert-augmented/checkpoint-1215"
# MODELNAME = "distilibert/checkpoint-750"
# MODELNAME = "distilibert-augmented/checkpoint-1215"

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(f"models/{MODELNAME}", device_map='cuda')
tokenizer = AutoTokenizer.from_pretrained(f"models/{MODELNAME}", device_map='cuda')

In [ ]:
ds = load_from_disk('dataset.hf')['test']

In [ ]:
tp = 0
tn = 0
fp = 0
fn = 0

guesses = [0,0]

for row in ds:
    prompt = row['text']
    label = row['label']

    with torch.no_grad():
        tokens = tokenizer(prompt, return_tensors='pt').to('cuda')
        tokenized_output = model(**tokens)
    
    guess = tokenized_output['logits'].argmax()
    guesses[guess] += 1
    if guess:
        if label:
            tp += 1
        else:
            fp += 1
    else:
        if label:
            fn += 1
        else:
            tn += 1

In [ ]:
print(f"{'Results':^20}")
print(f"Accuracy:   {(tp+tn)/len(ds):<10.2%}")
print(f"Precision:  {tp/(tp+fp):<10.2%}")
print(f"Recall:     {tp/(tp+fn):<10.2%}")
print(f"F1 Score:   {(2*tp)/(2*tp+fp+fn):<10.2%}")
print(f"Guesses:    [{guesses[0]}, {guesses[1]}]")